# MecaniQA - OAT 1: Compreensão e Baseline

Este Notebook organiza a série temporal, realiza a análise exploratória, trata dados ausentes e outliers, cria baselines e treina um Pipeline para prever as trocas de óleo do dia seguinte.

## Bibliotecas

Caso necessário, instale as dependências com: `py -3.14 -m pip install pandas matplotlib statsmodels openpyxl scikit-learn`

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from statsmodels.tsa.seasonal import seasonal_decompose

In [ ]:
ARQUIVO_DADOS = Path('datasets/mecaniqa_dataset.xlsx')

df = pd.read_excel(ARQUIVO_DADOS)
df['Data'] = pd.to_datetime(df['Data'])
df = df.sort_values('Data').set_index('Data').asfreq('D')

df.head()

## Inspeção inicial

A base possui a data e as métricas `Trocas_Oleo` e `Manutencao_Motor`. A frequência diária é definida para identificar dias sem registro.

In [ ]:
print(f'Período: {df.index.min().date()} até {df.index.max().date()}')
print(f'Registros: {len(df)}')
print(f'Datas duplicadas: {df.index.duplicated().sum()}')
display(df.info())
display(df.isna().sum().rename('Valores ausentes').to_frame())

## Limpeza e tratamento de outliers

Para a EDA, valores ausentes são interpolados no tempo. Outliers são limitados pelos limites do intervalo interquartil (IQR). Para o modelo, o tratamento de nulos e outliers é aprendido apenas no treino dentro do Pipeline.

In [ ]:
def tratar_serie_eda(serie):
    serie_preenchida = serie.astype(float).interpolate(method='time').ffill().bfill()
    q1, q3 = serie_preenchida.quantile([0.25, 0.75])
    iqr = q3 - q1
    limite_inferior = q1 - 1.5 * iqr
    limite_superior = q3 + 1.5 * iqr
    outliers = (serie_preenchida < limite_inferior) | (serie_preenchida > limite_superior)
    return serie_preenchida.clip(limite_inferior, limite_superior), int(outliers.sum())

trocas_oleo_eda, total_outliers_oleo = tratar_serie_eda(df['Trocas_Oleo'])
motor_eda, total_outliers_motor = tratar_serie_eda(df['Manutencao_Motor'])

print(f'Outliers detectados em Trocas_Oleo: {total_outliers_oleo}')
print(f'Outliers detectados em Manutencao_Motor: {total_outliers_motor}')

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(df.index, df['Trocas_Oleo'], label='Trocas de óleo originais', alpha=0.45)
ax.plot(trocas_oleo_eda.index, trocas_oleo_eda, label='Trocas de óleo tratadas', linewidth=2)
ax.set_title('Série temporal de trocas de óleo')
ax.set_xlabel('Data')
ax.set_ylabel('Quantidade')
ax.legend()
plt.tight_layout()
plt.show()

## Decomposição sazonal

Foi escolhido o modelo aditivo com período de 7 dias. Ele é apropriado para oscilações semanais de amplitude aproximadamente constante.

In [ ]:
decomposicao = seasonal_decompose(trocas_oleo_eda, model='additive', period=7)
fig = decomposicao.plot()
fig.set_size_inches(12, 8)
plt.tight_layout()
plt.show()

## Preparação para previsão

O alvo é a quantidade de trocas de óleo do dia seguinte. Os primeiros 80% dos dados são treino e os 20% finais são teste, preservando a ordem temporal.

In [ ]:
dados_modelo = df[['Trocas_Oleo', 'Manutencao_Motor']].copy()
dados_modelo['Demanda_Amanha'] = dados_modelo['Trocas_Oleo'].shift(-1)
dados_modelo = dados_modelo.dropna(subset=['Demanda_Amanha'])

colunas_entrada = ['Trocas_Oleo', 'Manutencao_Motor']
X = dados_modelo[colunas_entrada]
y = dados_modelo['Demanda_Amanha']

ponto_corte = int(len(dados_modelo) * 0.8)
X_train, X_test = X.iloc[:ponto_corte], X.iloc[ponto_corte:]
y_train, y_test = y.iloc[:ponto_corte], y.iloc[ponto_corte:]

print(f'Treino: {len(X_train)} registros | Teste: {len(X_test)} registros')

In [ ]:
class IQRClipper(BaseEstimator, TransformerMixin):
    """Limita outliers usando limites aprendidos apenas no treino."""

    def fit(self, X, y=None):
        X_array = np.asarray(X, dtype=float)
        q1 = np.nanpercentile(X_array, 25, axis=0)
        q3 = np.nanpercentile(X_array, 75, axis=0)
        iqr = q3 - q1
        self.limite_inferior_ = q1 - 1.5 * iqr
        self.limite_superior_ = q3 + 1.5 * iqr
        return self

    def transform(self, X):
        X_array = np.asarray(X, dtype=float)
        return np.clip(X_array, self.limite_inferior_, self.limite_superior_)

## Pipeline sem vazamento de dados

O `fit` é aplicado somente a `X_train` e `y_train`. Assim, imputação, limites de outliers, escala e modelo não usam informações do período de teste.

In [ ]:
pipeline = Pipeline(steps=[
    ('imputacao', SimpleImputer(strategy='median')),
    ('outliers', IQRClipper()),
    ('padronizacao', StandardScaler()),
    ('modelo', RandomForestRegressor(n_estimators=200, random_state=42))
])

pipeline.fit(X_train, y_train)
previsao_pipeline = pd.Series(pipeline.predict(X_test), index=y_test.index, name='Pipeline')
print('Pipeline treinado com sucesso.')

## Modelos baseline

O modelo Naive prevê que amanhã terá a mesma demanda de hoje. As médias móveis de 7 e 30 dias usam a média dos períodos anteriores como previsão.

In [ ]:
previsao_naive = X_test['Trocas_Oleo'].ffill().bfill().rename('Naive')
previsao_ma7 = df['Trocas_Oleo'].rolling(7, min_periods=1).mean().reindex(X_test.index).ffill().bfill().rename('Média móvel 7 dias')
previsao_ma30 = df['Trocas_Oleo'].rolling(30, min_periods=1).mean().reindex(X_test.index).ffill().bfill().rename('Média móvel 30 dias')

metricas = pd.DataFrame({
    'Modelo': ['Naive', 'Média móvel 7 dias', 'Média móvel 30 dias', 'Pipeline Random Forest'],
    'MAE': [
        mean_absolute_error(y_test, previsao_naive),
        mean_absolute_error(y_test, previsao_ma7),
        mean_absolute_error(y_test, previsao_ma30),
        mean_absolute_error(y_test, previsao_pipeline),
    ],
}).sort_values('MAE').reset_index(drop=True)

metricas

In [ ]:
datas_alvo = y_test.index + pd.Timedelta(days=1)

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(datas_alvo, y_test.values, label='Valores reais', color='black', linewidth=2)
ax.plot(datas_alvo, previsao_naive.values, label='Naive', alpha=0.8)
ax.plot(datas_alvo, previsao_ma7.values, label='Média móvel 7 dias', alpha=0.8)
ax.plot(datas_alvo, previsao_ma30.values, label='Média móvel 30 dias', alpha=0.8)
ax.plot(datas_alvo, previsao_pipeline.values, label='Pipeline Random Forest', linewidth=2)
ax.set_title('Valores reais e previsões de trocas de óleo')
ax.set_xlabel('Data da previsão')
ax.set_ylabel('Trocas de óleo')
ax.legend(ncol=2)
plt.tight_layout()
plt.show()

## Conclusão

O Notebook organiza e limpa a base temporal, mostra tendência, sazonalidade e ruído, compara os baselines com o Pipeline e mede o erro de cada abordagem.